# 12. 객체 탐지의 초기 흐름

이 노트북은 YOLO로 바로 들어가기 전에, 객체 탐지가 어떤 방식으로 발전했는지 최소한의 흐름을 정리합니다.

핵심 질문은 단순합니다.

- 이미지 안의 모든 위치를 다 검사해야 할까?
- 후보 영역을 먼저 고른 뒤 분류기를 적용하면 어떨까?
- 왜 R-CNN 계열은 중요했고, 왜 더 빠른 one-stage detector가 필요해졌을까?

이번 노트북의 목표는 다음과 같습니다.

- sliding window 방식의 직관과 한계를 이해합니다.
- region proposal이 왜 등장했는지 설명할 수 있습니다.
- R-CNN, Fast R-CNN, Faster R-CNN의 차이를 큰 흐름으로 정리합니다.
- YOLO가 등장한 배경을 자연스럽게 연결합니다.


In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['axes.unicode_minus'] = False


## 12-1. 분류기를 탐지기로 바꾸는 가장 단순한 생각

이미지 분류기는 보통 이미지 전체를 보고 `cat`, `dog`, `car` 같은 클래스를 출력합니다. 그렇다면 작은 창(window)을 이미지 위에서 움직이면서 각 위치를 분류하면 탐지가 가능할 것처럼 보입니다.

이 방식이 **sliding window** 입니다.

아이디어는 단순하지만, 이미지 크기와 window 크기, stride 조합에 따라 검사할 후보가 폭발적으로 늘어납니다.


In [ ]:
def count_windows(image_size, window_size, stride):
    image_w, image_h = image_size
    win_w, win_h = window_size
    count_x = ((image_w - win_w) // stride) + 1
    count_y = ((image_h - win_h) // stride) + 1
    return max(0, count_x) * max(0, count_y)


image_size = (640, 480)
window_sizes = [(64, 64), (128, 128), (256, 256)]
strides = [16, 32, 64]

for stride in strides:
    total = 0
    print(f'--- stride={stride} ---')
    for window_size in window_sizes:
        num_windows = count_windows(image_size, window_size, stride)
        total += num_windows
        print(f'window={window_size}: {num_windows} candidates')
    print('total:', total)


## 12-2. Sliding window 시각화

아래 그림은 window가 이미지 위를 일정 간격으로 이동하는 모습을 단순화한 것입니다. 실제로는 여러 크기와 비율의 window를 동시에 써야 하므로 후보 수가 훨씬 많아집니다.


In [ ]:
def draw_scene(ax):
    ax.set_xlim(0, 240)
    ax.set_ylim(180, 0)
    ax.set_facecolor('#eef6ff')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.add_patch(Rectangle((70, 55), 70, 60, color='#f5b76e', alpha=0.95))
    ax.text(88, 90, 'cat', fontsize=12, weight='bold')


fig, ax = plt.subplots(figsize=(7, 5))
draw_scene(ax)
ax.set_title('Sliding window 후보들')

for y in range(20, 130, 35):
    for x in range(20, 170, 35):
        ax.add_patch(Rectangle((x, y), 60, 50, fill=False, edgecolor='royalblue', linewidth=1, alpha=0.55))

plt.show()


## 12-3. Sliding window의 한계

Sliding window는 개념적으로 이해하기 쉽지만 실전에서는 비효율적입니다.

- 위치가 많습니다.
- 크기와 종횡비를 여러 개 써야 합니다.
- 각 후보마다 분류기를 반복 실행하면 계산량이 큽니다.
- 객체 모양과 정확히 맞는 window를 고르기 어렵습니다.

그래서 모든 window를 다 검사하기보다, 객체가 있을 만한 후보 영역만 먼저 뽑자는 접근이 등장했습니다.


## 12-4. Region proposal

Region proposal은 이미지에서 객체가 있을 법한 영역을 먼저 제안하는 과정입니다. 초기 방식에서는 Selective Search 같은 알고리즘이 많이 사용되었습니다.

분류기가 모든 위치를 다 보지 않고 후보 영역만 보면 계산량을 줄일 수 있습니다. 하지만 proposal을 만드는 과정 자체가 느리거나, 딥러닝 모델과 끝까지 함께 학습되지 않는다는 문제가 남았습니다.


In [ ]:
proposals = [
    (62, 45, 150, 125, 'proposal 1'),
    (25, 20, 105, 95, 'proposal 2'),
    (125, 80, 215, 160, 'proposal 3'),
    (68, 52, 140, 115, 'proposal 4'),
]

fig, ax = plt.subplots(figsize=(7, 5))
draw_scene(ax)
ax.set_title('Region proposal 예시')
colors = ['crimson', 'royalblue', 'seagreen', 'darkorange']
for proposal, color in zip(proposals, colors):
    x1, y1, x2, y2, label = proposal
    ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2))
    ax.text(x1, y1 - 5, label, color=color, fontsize=10, weight='bold')

plt.show()


## 12-5. R-CNN의 기본 흐름

R-CNN은 region proposal을 딥러닝 분류기와 연결한 대표적인 초기 탐지 방식입니다.

흐름은 다음과 같습니다.

1. 이미지에서 region proposal을 약 2,000개 생성합니다.
2. 각 proposal 영역을 잘라서 같은 크기로 변환합니다.
3. CNN으로 feature를 추출합니다.
4. 분류기와 bounding box regressor로 클래스와 위치 보정을 수행합니다.

문제는 proposal마다 CNN을 반복 실행해야 한다는 점입니다. 같은 이미지에서 겹치는 영역이 많아도 계산을 계속 반복합니다.


In [ ]:
steps = [
    ('Image', 0.05, 0.55),
    ('Region\nProposals', 0.27, 0.55),
    ('CNN per\nProposal', 0.50, 0.55),
    ('Class +\nBox', 0.73, 0.55),
]

fig, ax = plt.subplots(figsize=(9, 3))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('R-CNN: proposal마다 CNN 반복')

for text, x, y in steps:
    ax.add_patch(Rectangle((x, y - 0.15), 0.17, 0.3, fill=True, facecolor='#e0f2fe', edgecolor='#0369a1', linewidth=2))
    ax.text(x + 0.085, y, text, ha='center', va='center', fontsize=11, weight='bold')

for _, x, y in steps[:-1]:
    ax.annotate('', xy=(x + 0.22, y), xytext=(x + 0.17, y), arrowprops={'arrowstyle': '->', 'linewidth': 2})

plt.show()


## 12-6. Fast R-CNN과 Faster R-CNN

R-CNN의 비효율을 줄이기 위해 Fast R-CNN은 CNN을 이미지 전체에 한 번만 적용합니다. 그 다음 proposal 영역에 해당하는 feature만 잘라서 사용합니다.

Faster R-CNN은 한 단계 더 나아가 region proposal도 신경망 내부에서 생성합니다. 이때 proposal을 만드는 작은 네트워크를 **RPN(Region Proposal Network)** 이라고 부릅니다.

큰 흐름은 다음처럼 정리할 수 있습니다.

- R-CNN: proposal마다 CNN 실행, 느림
- Fast R-CNN: 이미지 전체 CNN 1회, proposal feature 공유
- Faster R-CNN: proposal 생성까지 신경망 안으로 통합

이 계열은 보통 two-stage detector라고 부릅니다. 먼저 후보를 만들고, 그 후보를 다시 분류하고 보정하기 때문입니다.


In [ ]:
rows = [
    ('R-CNN', '외부 proposal', 'proposal마다 CNN', '느림'),
    ('Fast R-CNN', '외부 proposal', 'CNN feature 공유', '개선'),
    ('Faster R-CNN', 'RPN', 'CNN feature 공유', '더 빠름'),
    ('YOLO', 'proposal 단계 없음', '한 번에 예측', '실시간 지향'),
]

header = f"{'모델':<16} | {'후보 생성':<16} | {'특징 추출':<18} | {'속도 관점':<10}"
print(header)
print('-' * len(header))
for row in rows:
    print(f'{row[0]:<16} | {row[1]:<16} | {row[2]:<18} | {row[3]:<10}')


## 12-7. Two-stage와 one-stage

객체 탐지 모델은 큰 관점에서 two-stage와 one-stage로 나눌 수 있습니다.

- two-stage detector: 후보 영역을 먼저 만들고, 각 후보를 다시 분류하고 보정합니다.
- one-stage detector: 이미지나 feature map 위에서 박스와 클래스를 한 번에 예측합니다.

Faster R-CNN은 대표적인 two-stage detector입니다. YOLO, SSD, RetinaNet은 대표적인 one-stage detector입니다.

YOLO는 `You Only Look Once`라는 이름처럼 이미지를 한 번 보고 바로 여러 박스와 클래스를 예측하는 방향을 취합니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, title, boxes in [
    (axes[0], 'Two-stage: 후보를 먼저 만들고 분류', ['Image', 'Proposals', 'Class + Box']),
    (axes[1], 'One-stage: 한 번에 박스와 클래스 예측', ['Image', 'Dense Predictions']),
]:
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title(title)
    xs = [0.08, 0.40, 0.70][:len(boxes)]
    for text, x in zip(boxes, xs):
        ax.add_patch(Rectangle((x, 0.38), 0.22, 0.24, facecolor='#fef3c7', edgecolor='#92400e', linewidth=2))
        ax.text(x + 0.11, 0.50, text, ha='center', va='center', fontsize=10, weight='bold')
    for x in xs[:-1]:
        ax.annotate('', xy=(x + 0.31, 0.50), xytext=(x + 0.22, 0.50), arrowprops={'arrowstyle': '->', 'linewidth': 2})

plt.tight_layout()
plt.show()


## 12-8. 왜 YOLO로 넘어가는가?

R-CNN 계열은 정확도와 구조 면에서 매우 중요하지만, 실시간 처리에는 부담이 큽니다. 특히 영상 탐지나 로봇, 자율주행처럼 빠른 반응이 필요한 상황에서는 속도가 중요합니다.

YOLO의 핵심 방향은 다음과 같습니다.

- 이미지 전체를 한 번에 봅니다.
- grid별로 박스와 클래스를 동시에 예측합니다.
- 별도의 proposal 단계를 최소화합니다.
- 후처리로 confidence threshold와 NMS를 적용합니다.

이제 다음 노트북에서 YOLO가 이미지를 grid로 나누고 각 grid에서 무엇을 예측하는지 살펴봅니다.


## 정리

- Sliding window는 분류기를 탐지기로 바꾸는 가장 단순한 접근이지만 후보 수가 많아 비효율적입니다.
- Region proposal은 객체가 있을 법한 후보 영역만 먼저 고르는 방식입니다.
- R-CNN은 proposal과 CNN 분류기를 결합했지만 proposal마다 CNN을 반복해 느렸습니다.
- Fast R-CNN은 feature를 공유했고, Faster R-CNN은 proposal 생성까지 신경망 내부로 가져왔습니다.
- YOLO는 후보 생성과 재분류 단계를 줄이고 한 번에 dense prediction을 수행하는 one-stage detector입니다.

다음 노트북 `13_YOLO_핵심_아이디어.ipynb`에서는 YOLO의 grid, box prediction, class score 개념을 직접 시각화합니다.
